# Access the analysis ready ORAS5 data

This notebook provides an example of how to open the reanalysis-oras5 Zarr store using xarray.
You will need to insert your CDS API key where indicated in the following code cell (available on [your profile page](https://cds-dev-cci2.copernicus-climate.eu/profile)).
For more information on using the Data Store Analysis Ready Datasets, please see the [user documentation pages](https://cds-dev-cci2.copernicus-climate.eu/datasets/how-to-use-the-dss-arco-dataset).

In [1]:
import os
cdsapi_key = "<INSERT-CDS-API-KEY-HERE>"

# The following attempts to find the CDSAPI key in your environment.
if cdsapi_key == "<INSERT-CDS-API-KEY-HERE>":
    cdsapi_key = None
if cdsapi_key is None:
    cdsapi_key = os.getenv("CDSAPI_KEY")
if cdsapi_key is None and os.path.exists(os.path.expanduser("~/.cdsapirc")):
    with open(os.path.expanduser("~/.cdsapirc"), "r") as f:
        for line in f:
            if line.startswith("key:"):
                cdsapi_key = line.split(":")[1].strip()
                break
if cdsapi_key is None:
    raise ValueError("CDSAPI key not found. Please set the CDSAPI_KEY environment variable or create a ~/.cdsapirc file.")


## Plug and play access

The code below provides a simple plug and play example of how to use the Zarr Store for light-weight access.

In [2]:
import xarray as xr
from obstore.store import HTTPStore
from zarr.storage import ObjectStore

# Time-chunked store is optimised for spatial subsetting across a region.
# 2024 is recent data, so use the OPERATIONAL stream (consolidated ends earlier).
# obstore uses bundled CA roots (rustls), avoiding macOS SSL cert issues.
timechunked_operational_url = "https://arco.datastores.ecmwf.int/cadl-arco-time-001/arco/reanalysis_oras5/operational/timeChunked.zarr"

http_store = HTTPStore(
    timechunked_operational_url,
    client_options={"default_headers": {"Authorization": f"Bearer {cdsapi_key}"}},
)
store = ObjectStore(http_store, read_only=True)

ds = xr.open_zarr(store)

# Inspect the variables and coordinates
ds


<xarray.Dataset> Size: 51GB
Dimensions:    (time: 137, latitude: 721, longitude: 1440, elevation: 75)
Coordinates:
  * time       (time) datetime64[ns] 1kB 2015-01-01 2015-02-01 ... 2026-05-01
  * latitude   (latitude) float64 6kB -90.0 -89.75 -89.5 ... 89.5 89.75 90.0
  * longitude  (longitude) float64 12kB -180.0 -179.8 -179.5 ... 179.5 179.8
  * elevation  (elevation) float32 300B -5.902e+03 -5.698e+03 ... -1.556 -0.5058
Data variables: (12/15)
    iicethic   (time, latitude, longitude) float32 569MB ...
    iicevelu   (time, latitude, longitude) float32 569MB ...
    iicevelv   (time, latitude, longitude) float32 569MB ...
    ileadfra   (time, latitude, longitude) float32 569MB ...
    so20chgt   (time, latitude, longitude) float32 569MB ...
    sohtc300   (time, latitude, longitude) float32 569MB ...
    ...         ...
    somxl030   (time, latitude, longitude) float32 569MB ...
    sosaline   (time, latitude, longitude) float32 569MB ...
    sossheig   (time, latitude, longitude) float32 569MB ...
    sosstsst   (time, latitude, longitude) float32 569MB ...
    sozotaux   (time, latitude, longitude) float32 569MB ...
    vosaline   (time, elevation, latitude, longitude) float32 43GB ...

## Subset to the ERA5 region & grid, then export CSV + metadata

Select the three variables for 2015–2026, subset to the same El Nino Pacific region as the ERA5 dataset, interpolate onto the same 2-degree grid, and write a CSV plus a metadata JSON.


In [ ]:
import os
import json
import numpy as np

DATA_DIR = '../../data/ORAS5'
os.makedirs(DATA_DIR, exist_ok=True)

# Year range to download
START_YEAR = 2015
END_YEAR = 2026

# Requested variables (ORAS5 / NEMO short names)
VARS = [
    'so20chgt',   # depth_of_20_c_isotherm
    'sohtc300',   # ocean_heat_content_for_the_upper_300m
    'sossheig',   # sea_surface_height
]

# Select the three variables over the year range (monthly steps)
sub = ds[VARS].sel(time=slice(f'{START_YEAR}', f'{END_YEAR}'))

# Same region as the ERA5 dataset (El Nino Pacific box). Longitudes here are
# -180..180 and the region crosses the dateline, so convert to 0-360 first.
sub = sub.assign_coords(longitude=(sub.longitude % 360)).sortby('longitude')
sub = sub.sel(latitude=slice(-30, 30), longitude=slice(120, 280))

# Same 2-degree grid as ERA5 (area [30, 120, -30, -80], grid [2.0, 2.0])
target_lat = np.arange(-30, 30 + 2, 2.0)
target_lon = np.arange(120, 280 + 2, 2.0)
oras = sub.interp(latitude=target_lat, longitude=target_lon)

# Trigger the remote reads and bring the subset into memory
oras = oras.load()

# Save metadata
metadata = {
    'dimensions': dict(oras.sizes),
    'variables': {},
    'global_attributes': dict(oras.attrs),
}
for var in oras.variables:
    metadata['variables'][var] = {
        'dims': list(oras[var].dims),
        'shape': list(oras[var].shape),
        'dtype': str(oras[var].dtype),
        'attributes': dict(oras[var].attrs),
    }

meta_path = os.path.join(DATA_DIR, f'oras5_{START_YEAR}_{END_YEAR}_metadata.json')
with open(meta_path, 'w') as f:
    json.dump(metadata, f, indent=2, default=str)
print(f"Metadata saved to {meta_path}")

# Convert to DataFrame and save as CSV
csv_path = os.path.join(DATA_DIR, f'oras5_{START_YEAR}_{END_YEAR}.csv')
df = oras.to_dataframe().reset_index()
df.to_csv(csv_path, index=False)
print(f"CSV saved to {csv_path} ({len(df)} rows)")


Metadata saved to ../../data/ORAS5/oras5_2024_metadata.json
CSV saved to ../../data/ORAS5/oras5_2024.csv (30132 rows)


## Upload to Google Drive

Create an `ORAS` folder under the shared Drive parent and upload only the two derived files: the CSV and the metadata JSON (not the raw data).


In [ ]:
import os
import pickle
import mimetypes
from google_auth_oauthlib.flow import InstalledAppFlow
from google.auth.transport.requests import Request
from googleapiclient.discovery import build
from googleapiclient.http import MediaFileUpload

SCOPES = ['https://www.googleapis.com/auth/drive']
TOKEN_PATH = '../../token.pickle'
CLIENT_SECRETS = '../../client_secrets.json'

# Parent Drive folder under which all data folders are uploaded
DRIVE_PARENT_ID = '1zLJgkYkrM1LRgtl7v5sfAQ4aits9zfUq'

DATA_DIR = '../../data/ORAS5'

# Year range (must match the download cell above)
START_YEAR = 2015
END_YEAR = 2026

# Only upload the CSV and the metadata (two files)
FILES_TO_UPLOAD = [
    os.path.join(DATA_DIR, f'oras5_{START_YEAR}_{END_YEAR}.csv'),
    os.path.join(DATA_DIR, f'oras5_{START_YEAR}_{END_YEAR}_metadata.json'),
]


def get_drive_service():
    creds = None
    if os.path.exists(TOKEN_PATH):
        with open(TOKEN_PATH, 'rb') as f:
            creds = pickle.load(f)
    if not creds or not creds.valid:
        if creds and creds.expired and creds.refresh_token:
            creds.refresh(Request())
        else:
            flow = InstalledAppFlow.from_client_secrets_file(CLIENT_SECRETS, SCOPES)
            creds = flow.run_local_server(port=0)
        with open(TOKEN_PATH, 'wb') as f:
            pickle.dump(creds, f)
    return build('drive', 'v3', credentials=creds)


def get_or_create_folder(service, name, parent_id):
    """Return the Drive folder id for `name` under `parent_id`, creating it if needed."""
    query = (
        f"name = '{name}' and mimeType = 'application/vnd.google-apps.folder' "
        f"and '{parent_id}' in parents and trashed = false"
    )
    response = service.files().list(q=query, spaces='drive', fields='files(id, name)').execute()
    files = response.get('files', [])
    if files:
        return files[0]['id']
    folder = service.files().create(
        body={'name': name, 'mimeType': 'application/vnd.google-apps.folder', 'parents': [parent_id]},
        fields='id'
    ).execute()
    return folder['id']


service = get_drive_service()

# Create (or reuse) the ORAS folder to hold the CSV and metadata
oras_folder_id = get_or_create_folder(service, 'ORAS', DRIVE_PARENT_ID)
print(f"Folder: ORAS (Drive id: {oras_folder_id})")

for path in FILES_TO_UPLOAD:
    name = os.path.basename(path)
    mimetype = mimetypes.guess_type(path)[0] or 'application/octet-stream'
    media = MediaFileUpload(path, mimetype=mimetype, resumable=True)
    result = service.files().create(
        body={'name': name, 'parents': [oras_folder_id]},
        media_body=media,
        fields='id, name'
    ).execute()
    print(f"  Uploaded: {result['name']} (id: {result['id']})")

print("Done.")


Folder: ORAS (Drive id: 155COy8_s9dO2WXtDAF1R8_A0yahmL8VO)
  Uploaded: oras5_2024.csv (id: 1dgCwitm9c4lLdXoOYMRE25uU4E_3JvbF)
  Uploaded: oras5_2024_metadata.json (id: 1ZWBUZDAH2lG787o-1Xq6CVwdgwO_5kyH)
Done.


In [5]:
import os
import numpy as np
import pandas as pd

# Verify the ORAS5 output matches the ERA5 grid & region
oras_csv = '../../data/ORAS5/oras5_2024.csv'
era5_csv = '../../data/ERA5/era5_singlelevels_1980.csv'

o = pd.read_csv(oras_csv)
print("ORAS5 columns:", list(o.columns))
print("rows:", len(o))

olat = np.sort(o['latitude'].unique())
olon = np.sort(o['longitude'].unique())
otime = sorted(o['time'].unique())
print(f"latitudes : {olat.min()}..{olat.max()} step {np.unique(np.diff(olat))} ({olat.size} pts)")
print(f"longitudes: {olon.min()}..{olon.max()} step {np.unique(np.diff(olon))} ({olon.size} pts)")
print(f"time steps: {len(otime)} ({otime[0]} .. {otime[-1]})")

# Cross-check the grid against ERA5 (longitudes compared in 0-360)
if os.path.exists(era5_csv):
    e = pd.read_csv(era5_csv)
    elat = np.sort(e['latitude'].unique())
    elon = np.sort((e['longitude'].unique() % 360))
    print("\nGrid matches ERA5:")
    print("  latitudes match :", np.allclose(olat, elat))
    print("  longitudes match:", np.allclose(olon, elon))

print("\nExpected rows = 31 lat x 81 lon x 12 months =", 31 * 81 * 12)


ORAS5 columns: ['time', 'latitude', 'longitude', 'so20chgt', 'sohtc300', 'sossheig']
rows: 30132
latitudes : -30.0..30.0 step [2.] (31 pts)
longitudes: 120.0..280.0 step [2.] (81 pts)
time steps: 12 (2024-01-01 .. 2024-12-01)

Grid matches ERA5:
  latitudes match : True
  longitudes match: True

Expected rows = 31 lat x 81 lon x 12 months = 30132


This mirrors the ERA5 workflow: same region, same 2-degree grid, exported to CSV + metadata, with only those two files uploaded to the `ORAS` Drive folder.
